In [1]:
import dill 
import torch 
import torch.nn as nn
import numpy as np
from four_room.shortest_path import find_all_action_values
from four_room.env import FourRoomsEnv
from four_room.wrappers import gym_wrapper
import numpy as np
import imageio
import gymnasium as gym
from four_room.utils import obs_to_state
from four_room.arch import CNN
from tqdm import tqdm
import matplotlib.pyplot as plt
import random


import warnings
warnings.filterwarnings(action='once')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
%load_ext autoreload
%autoreload 2

from rnd_exploration.rnd import RNDNetwork

In [2]:
gym.register('MiniGrid-FourRooms-v1', FourRoomsEnv)
size = 19
with open('configs/train.pl', 'rb') as file:
    train_config = dill.load(file)

with open('configs/test_reachable.pl', 'rb') as file:
    test_config = dill.load(file)

with open('configs/validation_unreachable.pl', 'rb') as file:
    val_config = dill.load(file)

In [3]:
env = gym_wrapper(gym.make(
        'MiniGrid-FourRooms-v1', 
        agent_pos= train_config['agent positions'],
        goal_pos = train_config['goal positions'],
        doors_pos = train_config['topologies'],
        agent_dir = train_config['agent directions'],
        size=size, 
        max_steps=1200, 
    ),
    original_obs=True
)

In [ ]:
%%capture cap --no-stderr
net = RNDNetwork(env, lr=1e-2, device=device)

for i in range(2):

    obs, _ = env.reset()
    done = False 
    valid_pos = env.get_wrapper_attr('valid_pos')
    
    for idx in range(len(valid_pos)):
        env.get_wrapper_attr('move_valid_pos')(idx)
        
        for _ in range(4): 
            obs, _, _, _, _ = env.step(1)
            state = obs_to_state(obs)
            agent_pos = state[:2]
            q = find_all_action_values(state[:2], state[2], state[3:5], state[5:], 0.99, size)
            action = np.array([1])
            rnd_value = net.get_error(obs, action)
            net.observe(obs, action)
            
            print(f'Context is {(i+1)%200:04d} | agent x: {agent_pos[0]:03d} | agent y: {agent_pos[1]:03d} | RND Val: {rnd_value:.5f}', end='\r')    
            
            
with open('rnd_vals_lr1e_2.txt', 'w') as f:
    f.write(cap.stdout)    